# Automated Dental Caries Detection from Intraoral Radiographs
## Clinical Diagnostic AI Pipeline: Semantic Segmentation with UNet
### End-to-End System: 40 Epochs Training for Caries UNet

---

### Executive Clinical Diagnostic Architecture

Automated dental diagnostics requires deep learning models tailored to the physical characteristics of oral pathology:

| Clinical Target | Physical Modality | Diagnostic Characteristic | AI Model Architecture |
| :--- | :--- | :--- | :--- |
| **Dental Caries (Cavities & Decay)** | **Intraoral Radiographs (Bitewing / Periapical X-rays)** | Radiolucent demineralization of enamel & dentin | **Caries UNet (ResNet-34)** trained with Balanced Sampler & Weighted BCE + Dice Loss (40 Epochs) |

---
### Notebook Structure
- **DENTAL CARIES DETECTION PIPELINE (Intraoral Radiographs)**
  - Section 1: Environment Setup, Dependencies & Hardware Acceleration
  - Section 2: Radiographic Data Ingestion & Polygon Rasterization Engine
  - Section 3: Exploratory Data Analysis & Tuned CLAHE Radiograph Enhancement
  - Section 4: Positive-Image Balanced Sampling (`WeightedRandomSampler`) & PyTorch Dataset
  - Section 5: Caries UNet Model Architecture & Compound Loss Formulation
  - Section 6: Caries Model 40-Epoch Training Pipeline & Best Checkpointing (`best_caries_unet.pth`)
  - Section 7: Dual-Tier Thresholding, Sensitivity-Precision Curves (ROC & PR-AUC)
  - Section 8: False Positive Inspection & Error Mapping
- **SECTION 9: Production Model Artifacts Verification for Streamlit Interactive UI**


In [ ]:
# ==============================================================================
# SECTION 1: Environment Setup, Dependencies & Hardware Acceleration
# ==============================================================================

import os
import sys
import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# Configure Matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] Active Hardware: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"[*] CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"[*] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Project Directory Structure
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebooks' else NOTEBOOK_DIR
DATASETS_DIR = ROOT_DIR / "Datasets"
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PERIAPICAL_DIR = DATASETS_DIR / "Intraoral Radiographic Database for Periapical Les"
ODS_DIR = DATASETS_DIR / "ODS_dataset"

print(f"[*] Root Workspace: {ROOT_DIR}")
print(f"[*] Periapical Radiographs Path: {PERIAPICAL_DIR.exists()} ({PERIAPICAL_DIR})")
print(f"[*] ODS Dataset Path: {ODS_DIR.exists()} ({ODS_DIR})")
print(f"[*] Models Directory: {MODELS_DIR}")


---
# PART A: DENTAL CARIES DETECTION PIPELINE (Intraoral Radiographs)

## Section 2: Radiographic Data Ingestion & Polygon Rasterization Engine

The radiographic dataset (`Datasets/Intraoral Radiographic Database for Periapical Les`) pairs 916 high-resolution radiographs with YOLO normalized polygon annotations:
- **Class 0: Dental Caries** (Demineralization of dental hard tissue / coronal tooth decay)
- **Class 1: Periapical Lesion** (Apical periodontitis / periapical abscess around root tips)
- **Class 2: Restoration** (Dental fillings / crowns)

We parse the normalized polygon coordinate stream and accurately rasterize Class 0 (`Caries`) into high-resolution binary segmentation masks.


In [ ]:
# ==============================================================================
# SECTION 2: Path Resolution & Polygon Rasterization Functions
# ==============================================================================

RAD_IMG_DIR = PERIAPICAL_DIR / "images"
RAD_LBL_DIR = PERIAPICAL_DIR / "labels"

rad_img_files = sorted(list(RAD_IMG_DIR.glob("*.jpg")) + list(RAD_IMG_DIR.glob("*.png")))
print(f"[*] Total radiographic images found: {len(rad_img_files)}")

# Match image to label file
radiograph_pairs = []
for img_p in rad_img_files:
    lbl_p = RAD_LBL_DIR / f"{img_p.stem}.txt"
    if lbl_p.exists():
        radiograph_pairs.append((str(img_p), str(lbl_p)))

print(f"[*] Total verified Radiograph-Annotation pairs: {len(radiograph_pairs)}")

def rasterize_yolo_polygons(label_path: str, img_shape: tuple, target_class: int = 0) -> np.ndarray:
    """
    Rasterizes normalized YOLO polygon coordinates into a binary segmentation mask.
    Handles utf-8-sig encoded text files.
    """
    h, w = img_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    if not os.path.exists(label_path):
        return mask
        
    with open(label_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 7:
                cls_id = int(parts[0])
                if cls_id == target_class:
                    coords = [float(x) for x in parts[1:]]
                    pts = []
                    for i in range(0, len(coords), 2):
                        px = int(round(coords[i] * w))
                        py = int(round(coords[i+1] * h))
                        pts.append([px, py])
                    if len(pts) >= 3:
                        poly_np = np.array(pts, dtype=np.int32)
                        cv2.fillPoly(mask, [poly_np], 1)
    return mask

# Verify class distribution across dataset
caries_positive_count = 0
total_caries_instances = 0

for _, lbl_path in radiograph_pairs:
    with open(lbl_path, 'r', encoding='utf-8-sig') as f:
        has_caries = False
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 7 and int(parts[0]) == 0:
                has_caries = True
                total_caries_instances += 1
        if has_caries:
            caries_positive_count += 1

print(f"[*] Caries-Positive Radiographs: {caries_positive_count} / {len(radiograph_pairs)} ({caries_positive_count/len(radiograph_pairs)*100:.1f}%)")
print(f"[*] Sound / Caries-Negative Radiographs: {len(radiograph_pairs) - caries_positive_count}")
print(f"[*] Total Caries Lesion Annotations: {total_caries_instances}")


---
## Section 3: Exploratory Data Analysis & Tuned CLAHE Preprocessing

Dental radiographs suffer from patient-to-patient exposure variation, cervical burnout, and alveolar bone noise.
Contrast Limited Adaptive Histogram Equalization (CLAHE) with `clipLimit=1.5` and `tileGridSize=(8, 8)` boosts subtle demineralization boundaries at the enamel-dentin junction without amplifying high-frequency trabecular noise.


In [ ]:
# ==============================================================================
# SECTION 3: Tuned CLAHE Radiograph Enhancement
# ==============================================================================

clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))

def preprocess_radiograph(img_bgr: np.ndarray, target_size=(512, 512)) -> tuple:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    enhanced = clahe.apply(gray)
    rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    resized_rgb = cv2.resize(rgb, target_size, interpolation=cv2.INTER_LINEAR)
    return resized_rgb

# Visualize Raw vs Enhanced on sample cavity case
sample_idx = 3 # patient_0004 has active cavity
s_img_p, s_lbl_p = radiograph_pairs[sample_idx]
s_bgr = cv2.imread(s_img_p)
s_mask = rasterize_yolo_polygons(s_lbl_p, s_bgr.shape, target_class=0)
s_enh = preprocess_radiograph(s_bgr)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(s_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Raw Radiograph")
axes[0].axis('off')

axes[1].imshow(s_enh)
axes[1].set_title("Tuned CLAHE Enhanced (clipLimit=1.5)")
axes[1].axis('off')

overlay = s_enh.copy()
overlay[cv2.resize(s_mask, (512, 512)) == 1] = [255, 30, 30]
axes[2].imshow(cv2.addWeighted(s_enh, 0.70, overlay, 0.30, 0))
axes[2].set_title(f"Ground Truth Caries Lesion ({np.sum(s_mask > 0)} px)")
axes[2].axis('off')
plt.tight_layout()
plt.show()


---
## Section 4: Positive-Image Balanced Sampling & Dataset Pipeline

In standard random training, 76% of batches are composed of sound images with all-zero masks, starving the network of positive gradients.
We introduce **`WeightedRandomSampler`**, assigning inverse class probabilities so that every batch achieves a **50/50 balance of positive and negative radiographs**.


In [ ]:
# ==============================================================================
# SECTION 4: PyTorch Dataset with Positive-Image Balanced Sampler
# ==============================================================================

print("[*] Pre-caching resized 512x512 CLAHE-enhanced radiographs and masks into RAM...")
t_cache = time.time()

def cache_radiograph(pair):
    img_p, lbl_p = pair
    bgr = cv2.imread(img_p)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    enhanced = clahe.apply(gray)
    rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    rgb_512 = cv2.resize(rgb, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask = rasterize_yolo_polygons(lbl_p, bgr.shape, target_class=0)
    mask_512 = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
    return rgb_512, mask_512

rad_cached_data = [cache_radiograph(p) for p in radiograph_pairs]
print(f"[*] Pre-cached {len(rad_cached_data)} radiographs in {time.time() - t_cache:.2f}s.")

# Train / Val / Test Split (70% Train, 15% Val, 15% Test)
np.random.seed(SEED)
indices = list(range(len(rad_cached_data)))
np.random.shuffle(indices)

n_total = len(indices)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_cached = [rad_cached_data[i] for i in indices[:n_train]]
val_cached = [rad_cached_data[i] for i in indices[n_train:n_train + n_val]]
test_cached = [rad_cached_data[i] for i in indices[n_train + n_val:]]

tr_pos = sum(1 for _, m in train_cached if np.sum(m) > 0)
val_pos = sum(1 for _, m in val_cached if np.sum(m) > 0)
test_pos = sum(1 for _, m in test_cached if np.sum(m) > 0)

print(f"[*] Caries Splits - Train: {len(train_cached)} ({tr_pos} positive), Val: {len(val_cached)} ({val_pos} positive), Test: {len(test_cached)} ({test_pos} positive)")

# Balanced Sampler for 50/50 Train Batches
sample_weights = []
w_pos = (len(train_cached) - tr_pos) / tr_pos if tr_pos > 0 else 1.0
w_neg = 1.0
for _, m in train_cached:
    sample_weights.append(w_pos if np.sum(m) > 0 else w_neg)
caries_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_cached), replacement=True)

class CachedRadiographDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        rgb, mask = self.data[idx]
        if self.transform:
            aug = self.transform(image=rgb, mask=mask)
            img = aug['image']
            mask = aug['mask']
        else:
            img = rgb.astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img = (img - mean) / std
            img = torch.tensor(img).permute(2, 0, 1).float()
            mask = torch.tensor(mask).float()
            
        if mask.dim() == 2:
            mask = mask.unsqueeze(0).float()
        else:
            mask = mask.float()
        return img, mask

caries_train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), rotate=(-10, 10), border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

caries_val_tf = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

caries_train_ds = CachedRadiographDataset(train_cached, transform=caries_train_tf)
caries_val_ds = CachedRadiographDataset(val_cached, transform=caries_val_tf)
caries_test_ds = CachedRadiographDataset(test_cached, transform=caries_val_tf)

caries_train_loader = DataLoader(caries_train_ds, batch_size=16, sampler=caries_sampler, pin_memory=True)
caries_val_loader = DataLoader(caries_val_ds, batch_size=16, shuffle=False, pin_memory=True)
caries_test_loader = DataLoader(caries_test_ds, batch_size=16, shuffle=False, pin_memory=True)


---
## Section 5: Caries UNet Model Architecture & Compound Loss Formulation (40 Epochs)

We formulate **Weighted BCE ($w_{\text{pos}}=4.0$) + Soft Dice Loss** to optimize boundary precision and set training to **40 Epochs** with Cosine Annealing learning rate scheduling.


In [ ]:
# ==============================================================================
# SECTION 5: Loss Functions & Model Instantiation (40 Epochs)
# ==============================================================================

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        intersection = (probs * targets_f).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets_f.sum() + self.smooth)
        return 1.0 - dice

class EnhancedCariesLoss(nn.Module):
    def __init__(self, pos_weight=4.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
        self.dice = SoftDiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

caries_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

caries_criterion = EnhancedCariesLoss(pos_weight=4.0)
caries_optimizer = optim.AdamW(caries_model.parameters(), lr=1.5e-4, weight_decay=1e-4)

CARIES_EPOCHS = 40
caries_scheduler = optim.lr_scheduler.CosineAnnealingLR(caries_optimizer, T_max=CARIES_EPOCHS, eta_min=1e-6)
caries_scaler = torch.amp.GradScaler('cuda')

print(f"[*] Caries UNet Initialized. Configured for {CARIES_EPOCHS} epochs with Cosine Annealing.")


---
## Section 6: Caries Model 40-Epoch Training Pipeline

Executes 40 epochs of training with validation Dice checkpointing. The highest validation Dice checkpoint across all epochs is automatically saved to `models/best_caries_unet.pth`.


In [ ]:
# ==============================================================================
# SECTION 6: Caries Model 40-Epoch Training Loop & Curve Plotting
# ==============================================================================

caries_best_dice = 0.0
best_caries_path = MODELS_DIR / "best_caries_unet.pth"

caries_history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_sens': []}

print(f"[*] Training Caries UNet for {CARIES_EPOCHS} epochs on {device}...")
start_time = time.time()

for epoch in range(1, CARIES_EPOCHS + 1):
    ep_t0 = time.time()
    caries_model.train()
    running_loss = 0.0
    for imgs, msks in caries_train_loader:
        imgs = imgs.to(device)
        msks = msks.to(device)
        caries_optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = caries_model(imgs)
            loss = caries_criterion(out, msks)
        caries_scaler.scale(loss).backward()
        caries_scaler.step(caries_optimizer)
        caries_scaler.update()
        running_loss += loss.item()

    caries_scheduler.step()

    # Validation
    caries_model.eval()
    val_loss = 0.0
    val_tp, val_fp, val_fn = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, msks in caries_val_loader:
            imgs = imgs.to(device)
            msks = msks.to(device)
            with torch.amp.autocast('cuda'):
                out = caries_model(imgs)
                loss = caries_criterion(out, msks)
            val_loss += loss.item()

            probs = torch.sigmoid(out)
            preds = (probs >= 0.50).float()
            val_tp += (preds * msks).sum().item()
            val_fp += (preds * (1 - msks)).sum().item()
            val_fn += ((1 - preds) * msks).sum().item()

    eps = 1e-6
    val_dice = (2 * val_tp + eps) / (2 * val_tp + val_fp + val_fn + eps)
    val_sens = (val_tp + eps) / (val_tp + val_fn + eps)
    dur = time.time() - ep_t0

    caries_history['train_loss'].append(running_loss / len(caries_train_loader))
    caries_history['val_loss'].append(val_loss / len(caries_val_loader))
    caries_history['val_dice'].append(val_dice)
    caries_history['val_sens'].append(val_sens)

    is_best = val_dice > caries_best_dice
    if is_best:
        caries_best_dice = val_dice
        torch.save(caries_model.state_dict(), best_caries_path)

    if epoch % 5 == 0 or epoch == 1 or is_best:
        marker = f" [NEW BEST -> {best_caries_path.name}]" if is_best else ""
        print(f"Epoch [{epoch:02d}/{CARIES_EPOCHS:02d}] ({dur:.1f}s) | Train Loss: {caries_history['train_loss'][-1]:.4f} | "
              f"Val Loss: {caries_history['val_loss'][-1]:.4f} | Val Dice: {val_dice:.4f} | Val Sens: {val_sens:.4f}{marker}")

print(f"[*] Caries training complete in {(time.time() - start_time)/60:.2f} mins. Best Val Dice: {caries_best_dice:.4f}")

# Plot Caries Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(1, CARIES_EPOCHS + 1), caries_history['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
ax1.plot(range(1, CARIES_EPOCHS + 1), caries_history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
ax1.set_title('Caries UNet: Training & Validation Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Compound Loss')
ax1.legend()

ax2.plot(range(1, CARIES_EPOCHS + 1), caries_history['val_dice'], label='Val Dice (DSC)', color='#2ca02c', lw=2)
ax2.plot(range(1, CARIES_EPOCHS + 1), caries_history['val_sens'], label='Val Sensitivity (Recall)', color='#d62728', lw=2, linestyle='--')
ax2.set_title(f'Caries UNet: Validation Performance (Peak Dice: {caries_best_dice:.4f})', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.legend()
plt.tight_layout()
plt.show()


---
## Section 7: Dual-Tier Thresholding, Sensitivity-Precision Curves (ROC & PR-AUC)

We evaluate the test set across a full threshold sweep:
- **Screening Mode (Threshold $\approx 0.35 - 0.45$)**: Sensitivity $> 85\%$, ideal for early cavity detection.
- **Confirmatory Mode (Threshold $\approx 0.50 - 0.60$)**: High precision for surgical restoration planning.

We also compute and plot the **Receiver Operating Characteristic (ROC)** curve and **Precision-Recall (PR)** curve across test radiographs.


In [ ]:
# ==============================================================================
# SECTION 7: Test Set Evaluation, Dual-Tier Sweep & ROC/PR Curves
# ==============================================================================

if best_caries_path.exists():
    caries_model.load_state_dict(torch.load(best_caries_path, map_location=device))
caries_model.eval()

caries_test_probs = []
caries_test_targets = []

with torch.no_grad():
    for imgs, msks in tqdm(caries_test_loader, desc="Evaluating Caries Test Set"):
        imgs = imgs.to(device)
        out = caries_model(imgs)
        # Test-Time Augmentation (Horizontal Flip)
        out_flip = torch.flip(caries_model(torch.flip(imgs, dims=[3])), dims=[3])
        probs = (torch.sigmoid(out) + torch.sigmoid(out_flip)) / 2.0
        caries_test_probs.append(probs.cpu().numpy())
        caries_test_targets.append(msks.numpy())

caries_test_probs = np.concatenate(caries_test_probs, axis=0)
caries_test_targets = np.concatenate(caries_test_targets, axis=0)

thresholds = [0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 0.80]
sweep_rows = []

for th in thresholds:
    tp_tot, fp_tot, fn_tot, tn_tot = 0, 0, 0, 0
    for i in range(len(caries_test_probs)):
        p = (caries_test_probs[i, 0] >= th).astype(np.uint8)
        cnts, _ = cv2.findContours(p, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        clean_p = np.zeros_like(p)
        for c in cnts:
            if cv2.contourArea(c) >= 40:
                cv2.drawContours(clean_p, [c], -1, 1, thickness=-1)
        
        t = caries_test_targets[i, 0].astype(np.uint8)
        tp_tot += np.sum((clean_p == 1) & (t == 1))
        fp_tot += np.sum((clean_p == 1) & (t == 0))
        fn_tot += np.sum((clean_p == 0) & (t == 1))
        tn_tot += np.sum((clean_p == 0) & (t == 0))
        
    eps = 1e-6
    d = (2 * tp_tot + eps) / (2 * tp_tot + fp_tot + fn_tot + eps)
    s = (tp_tot + eps) / (tp_tot + fn_tot + eps)
    prec = (tp_tot + eps) / (tp_tot + fp_tot + eps)
    spec = (tn_tot + eps) / (tn_tot + fp_tot + eps)
    iou = (tp_tot + eps) / (tp_tot + fp_tot + fn_tot + eps)
    
    sweep_rows.append({
        'Threshold': th,
        'Dice (DSC)': round(d, 4),
        'Sensitivity': round(s, 4),
        'Specificity': round(spec, 4),
        'Precision': round(prec, 4),
        'IoU': round(iou, 4)
    })

df_caries_metrics = pd.DataFrame(sweep_rows)
print("
[*] Caries Test Diagnostic Performance across Threshold Sweep:")
print(df_caries_metrics.to_string(index=False))

# Compute ROC and PR Curves on flattened test pixels (subsampled for speed)
flat_probs = caries_test_probs.flatten()[::10]
flat_targets = caries_test_targets.flatten()[::10].astype(np.uint8)

fpr, tpr, _ = roc_curve(flat_targets, flat_probs)
roc_auc = auc(fpr, tpr)

precisions, recalls, pr_thresholds = precision_recall_curve(flat_targets, flat_probs)
pr_auc = auc(recalls, precisions)

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 5))
ax_roc.plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'UNet (AUC = {roc_auc:.4f})')
ax_roc.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax_roc.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight='bold')
ax_roc.set_xlabel('False Positive Rate (1 - Specificity)')
ax_roc.set_ylabel('True Positive Rate (Sensitivity)')
ax_roc.legend(loc='lower right')
ax_roc.grid(True)

ax_pr.plot(recalls, precisions, color='#d62728', lw=2.5, label=f'UNet (PR-AUC = {pr_auc:.4f})')
ax_pr.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax_pr.set_xlabel('Recall (Diagnostic Sensitivity)')
ax_pr.set_ylabel('Precision')
ax_pr.legend(loc='upper right')
ax_pr.grid(True)
plt.tight_layout()
plt.show()


---
## Section 8: False Positive Inspection & Clinical Error Analysis

We inspect False Positives using an Error Map:
- 🟩 **Green (True Positive)**: Correctly identified cavity matching ground truth.
- 🟥 **Red (False Negative)**: Under-segmented / missed decay.
- 🟦 **Blue (False Positive)**: Over-segmented sound tooth, cervical burnout, or pulpal radiolucency.


In [ ]:
# ==============================================================================
# SECTION 8: False Positive Inspection & Error Mapping
# ==============================================================================

def inspect_caries_errors(test_ds, test_probs, test_targets, threshold=0.45, max_cases=3):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    fp_cases = []
    for i in range(len(test_probs)):
        p = (test_probs[i, 0] >= threshold).astype(np.uint8)
        t = test_targets[i, 0].astype(np.uint8)
        fp_pixels = np.sum((p == 1) & (t == 0))
        if fp_pixels > 150:
            fp_cases.append((i, fp_pixels))

    fp_cases.sort(key=lambda x: x[1], reverse=True)
    selected = fp_cases[:max_cases]
    print(f"[*] Found {len(fp_cases)} test cases with false positives. Visualizing top {len(selected)}...")

    fig, axes = plt.subplots(len(selected), 4, figsize=(18, 4.5 * len(selected)))
    if len(selected) == 1: axes = np.expand_dims(axes, 0)

    for row_idx, (idx, fp_px) in enumerate(selected):
        img_tensor, _ = test_ds[idx]
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_disp = np.clip((img_np * std + mean) * 255.0, 0, 255).astype(np.uint8)

        p = (test_probs[idx, 0] >= threshold).astype(np.uint8)
        t = test_targets[idx, 0].astype(np.uint8)

        axes[row_idx, 0].imshow(img_disp)
        axes[row_idx, 0].set_title(f"Test Image #{idx} (FP: {fp_px} px)")
        axes[row_idx, 0].axis('off')

        gt_overlay = img_disp.copy()
        gt_overlay[t == 1] = [0, 230, 0]
        axes[row_idx, 1].imshow(cv2.addWeighted(img_disp, 0.65, gt_overlay, 0.35, 0))
        axes[row_idx, 1].set_title("Radiologist Ground Truth")
        axes[row_idx, 1].axis('off')

        pred_overlay = img_disp.copy()
        pred_overlay[p == 1] = [255, 50, 50]
        axes[row_idx, 2].imshow(cv2.addWeighted(img_disp, 0.65, pred_overlay, 0.35, 0))
        axes[row_idx, 2].set_title(f"Model Prediction (th={threshold})")
        axes[row_idx, 2].axis('off')

        err = img_disp.copy()
        err[(p == 1) & (t == 1)] = [0, 255, 0] # TP: Green
        err[(p == 0) & (t == 1)] = [255, 0, 0] # FN: Red
        err[(p == 1) & (t == 0)] = [0, 100, 255] # FP: Blue
        axes[row_idx, 3].imshow(cv2.addWeighted(img_disp, 0.60, err, 0.40, 0))
        axes[row_idx, 3].set_title("Error Map: TP(Grn), FN(Red), FP(Blu)")
        axes[row_idx, 3].axis('off')

    plt.tight_layout()
    plt.show()

inspect_caries_errors(caries_test_ds, caries_test_probs, caries_test_targets, threshold=0.45)


---
## Section 9: Production Model Verification & Streamlit Integration

The model has been trained for **40 Epochs** and the best checkpoint has been verified and saved to `models/`:
1. `models/best_caries_unet.pth`: Caries UNet for Intraoral Radiographs

The Streamlit web application connects the model dynamically:
```bash
.\.venv\Scripts\streamlit run app\app.py
```


In [ ]:
# ==============================================================================
# SECTION 9: Production Model Verification & Streamlit Launch Test
# ==============================================================================

print("=" * 80)
print("🎉 CLINICAL CARIES DIAGNOSTIC AI SUITE VERIFICATION")
print("=" * 80)

for m_name, m_path in [("Caries Radiograph UNet", best_caries_path)]:
    if m_path.exists():
        size_mb = m_path.stat().st_size / (1024 * 1024)
        print(f"[✓] {m_name:25s} | Exists: True | Size: {size_mb:6.2f} MB | Path: {m_path.relative_to(ROOT_DIR)}")
    else:
        print(f"[✗] {m_name:25s} | Exists: FALSE | Path: {m_path}")

print("\n[*] Streamlit Interactive UI Instructions:")
print("    Launch the diagnostic dashboard by running:")
print("    >> .\\.venv\\Scripts\\streamlit run app\\app.py")
print("=" * 80)
